# Čišćenje AllMusic dataseta (`allmusic.csv`)

Ovaj notebook čisti podatke prikupljene web scrapingom sa sajta AllMusic, pre nego što se spoje sa ostalim podacima.

Rešeni problemi:

1. **`album_title`** – uklanjanje imena izvođača koje se "zalepilo" na kraj naziva albuma.
2. **`allmusic_*_genre` / `allmusic_*_style`** – sečenje procurelog HTML/UI teksta ("đubreta") koji je ostao iza stvarnih žanrova/stilova.
3. **`allmusic_album_theme`** – zamena vrednosti koje su zapravo iz gornjeg navigacionog menija (Blues/Classical/Electronic…) sa `"Unknown"`.
4. **`allmusic_composers` / `allmusic_song_style` / `allmusic_album_mood` / `allmusic_album_theme`** – rekonstrukcija liste vrednosti koje su razdvojene samo razmakom (umesto zarezom), pomoću pristupa zasnovanog na rečniku poznatih vrednosti (vocabulary-based reconstruction).
5. **Dodatne izmene** – uklanjanje kolona koje su potpuno prazne/konstantne, normalizacija trajanja albuma, čišćenje placeholder teksta, trim whitespace-a.
6. Uklanjanje `allmusic_artist_url` / `allmusic_song_url` / `allmusic_album_url`.
7. `allmusic_track_pick` -> pravi `bool`.
8. `allmusic_release_date` -> format `YYYY-MM-DD`.
9. Ručna ispravka `allmusic_composers` za 5 konkretnih pesama i brisanje pomoćne kolone `allmusic_composers_confidence`.
10. Normalizacija `allmusic_recording_location` (uklanjanje datuma koji su greškom završili u ovoj koloni, spajanje varijanti istog mesta).
11. Popunjavanje preostalih nedostajućih vrednosti u tekstualnim kolonama sa `"Unknown"`.
12. Izvoz očišćenog dataseta u `allmusic_clean.csv`.

In [134]:
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 160)

## 0. Učitavanje i početni pregled podataka

In [135]:
df = pd.read_csv('allmusic.csv')
print('Dimenzije:', df.shape)
df.head(3)

Dimenzije: (2313, 24)


,artist_name,album_title,song_title,track_number,spotify_id,allmusic_artist_url,allmusic_song_url,allmusic_song_genre,allmusic_song_style,allmusic_song_mood,...,allmusic_album_genre,allmusic_album_style,allmusic_album_mood,allmusic_album_theme,allmusic_album_duration,allmusic_recording_location,allmusic_release_date,allmusic_match_status,allmusic_match_score,allmusic_match_type
0,The Verve,A Storm in Heaven The Verve,Star Sail,1,NaN,https://www.allmusic.com/artist/the-verve-mn0000575522,https://www.allmusic.com/song/star-sail-mt0053617233,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock Indie Rock Shoegaze,"Druggy, Reflective, Sensual, Trippy, Restrained, Stylish, Cerebral, Detached, Ethereal, Hypnotic, Nocturnal, Intimate, Melancholy",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, Dream Pop, Shoegaze, Space Rock, Britpop, Indie Pop, Indie Rock, Neo-Psychedelia, Noise Pop Listen on Amazon L...",Dreamy Spacey Cerebral Detached Restrained Trippy Bittersweet Brooding Cathartic Druggy Earnest Earthy Ethereal Hypnotic Intimate Melancholy Organic Passion...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
1,The Verve,A Storm in Heaven The Verve,Slide Away,2,NaN,https://www.allmusic.com/artist/the-verve-mn0000575522,https://www.allmusic.com/song/slide-away-mt0053617234,Pop/Rock,Alternative/Indie Rock Indie Rock Alternative Pop/Rock Space Rock Shoegaze,"Brash, Druggy, Reflective, Restrained, Stylish, Trippy, Cerebral, Detached, Nocturnal, Sensual, Ethereal, Hypnotic, Intimate, Melancholy",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, Dream Pop, Shoegaze, Space Rock, Britpop, Indie Pop, Indie Rock, Neo-Psychedelia, Noise Pop Listen on Amazon L...",Dreamy Spacey Cerebral Detached Restrained Trippy Bittersweet Brooding Cathartic Druggy Earnest Earthy Ethereal Hypnotic Intimate Melancholy Organic Passion...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
2,The Verve,A Storm in Heaven The Verve,Already There,3,NaN,https://www.allmusic.com/artist/the-verve-mn0000575522,https://www.allmusic.com/song/already-there-mt0053617235,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock Shoegaze Indie Rock,"Druggy, Reflective, Trippy, Restrained, Stylish, Cerebral, Detached, Ethereal, Hypnotic, Intimate",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, Dream Pop, Shoegaze, Space Rock, Britpop, Indie Pop, Indie Rock, Neo-Psychedelia, Noise Pop Listen on Amazon L...",Dreamy Spacey Cerebral Detached Restrained Trippy Bittersweet Brooding Cathartic Druggy Earnest Earthy Ethereal Hypnotic Intimate Melancholy Organic Passion...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN


In [136]:
print(df.dtypes)
print()
print('Broj nedostajućih vrednosti po koloni:')
print(df.isnull().sum())

artist_name                        str
album_title                        str
song_title                         str
track_number                     int64
spotify_id                     float64
allmusic_artist_url                str
allmusic_song_url                  str
allmusic_song_genre                str
allmusic_song_style                str
allmusic_song_mood                 str
allmusic_song_theme                str
allmusic_composers                 str
allmusic_track_pick                str
allmusic_album_url                 str
allmusic_album_genre               str
allmusic_album_style               str
allmusic_album_mood                str
allmusic_album_theme               str
allmusic_album_duration            str
allmusic_recording_location        str
allmusic_release_date              str
allmusic_match_status              str
allmusic_match_score           float64
allmusic_match_type            float64
dtype: object

Broj nedostajućih vrednosti po koloni:
artist_nam

Kolone `spotify_id`, `allmusic_match_score` i `allmusic_match_type` su **100% prazne**, a `allmusic_match_status` ima **jednu jedinu konstantnu vrednost** (`"pending"`) u svih 2313 redova.

## 1. `album_title` — uklanjanje "zalepljenog" izvođača

Provera pokazuje da je za (skoro) svaki album AllMusic naziv izvođača dodat na kraj naziva albuma, razdvojen razmakom, npr.:

```
artist_name = "The Verve"
album_title = "A Storm in Heaven The Verve"   ->  treba: "A Storm in Heaven"
```

Kod self-titled albuma (album se zove isto kao izvođač), ovo daje ponovljen naziv:

```
artist_name = "Blur"
album_title = "Blur Blur"                     ->  treba: "Blur"
```

Pravilo "skini sufiks ` ` + `artist_name`" ispravno rešava i ovaj slučaj (posle skidanja sufiksa ostaje tačno "Blur").

Postoji i par izuzetaka gde se na kraj naslova nije zalepio `artist_name` iz kolone, već **puna lista izvođača** sa AllMusic stranice albuma (kolaboracije, orkestarske obrade i sl.), ili gde uopšte nije bilo zalepljeno ništa. Te slučajeve rešavamo ručnim mapiranjem, pošto ih ima samo 5 i lako je proveriti svaki pojedinačno.

In [137]:
pairs = df[['artist_name', 'album_title']].drop_duplicates()

exceptions = []
for _, r in pairs.iterrows():
    a, t = r.artist_name, r.album_title
    if not t.endswith(' ' + a):
        exceptions.append((a, t))

print(f'Broj (izvođač, album) parova: {len(pairs)}')
print(f'Broj izuzetaka (ne završava se na " " + artist_name): {len(exceptions)}')
for a, t in exceptions:
    print(' -', repr(a), '|', repr(t))

Broj (izvođač, album) parova: 175
Broj izuzetaka (ne završava se na " " + artist_name): 4
 - 'Damon Albarn' | 'Mali Music Mali Music / Damon Albarn / Afel Bocoum / Toumani Diabaté'
 - 'Jarvis Cocker' | 'Room 29 Jarvis Cocker / Chilly Gonzales'
 - 'Liam Gallagher' | 'Liam Gallagher & John Squire Liam Gallagher / John Squire'
 - 'Paul Weller' | 'Jawbone'


In [138]:
# Ručno utvrđeni "čisti" naslovi za 5 izuzetaka (provereno ručno na AllMusic sajtu):
MANUAL_TITLE_OVERRIDES = {
    ('Damon Albarn', 'Mali Music Mali Music / Damon Albarn / Afel Bocoum / Toumani Diabaté'): 'Mali Music',
    ('Jarvis Cocker', 'Room 29 Jarvis Cocker / Chilly Gonzales'): 'Room 29',
    ('Liam Gallagher', 'Liam Gallagher & John Squire Liam Gallagher / John Squire'): 'Liam Gallagher & John Squire',
    ('Paul Weller', 'Jawbone'): 'Jawbone',  # već čist, ništa nije zalepljeno
    ('Paul Weller', 'An Orchestrated Songbook BBC Symphony Orchestra / Jules Buckley / Paul Weller'): 'An Orchestrated Songbook',
}

def clean_album_title(row):
    artist, title = row['artist_name'], row['album_title']
    key = (artist, title)
    if key in MANUAL_TITLE_OVERRIDES:
        return MANUAL_TITLE_OVERRIDES[key]
    suffix = ' ' + artist
    if title.endswith(suffix):
        return title[: -len(suffix)].strip()
    return title.strip()

df['album_title_original'] = df['album_title']  # čuvamo original radi provere
df['album_title'] = df.apply(clean_album_title, axis=1)

# provera
check = df[['artist_name', 'album_title_original', 'album_title']].drop_duplicates()
print(check.to_string(index=False))

           artist_name                                                          album_title_original                                                   album_title
             The Verve                                                   A Storm in Heaven The Verve                                             A Storm in Heaven
             The Verve                                                     A Northern Soul The Verve                                               A Northern Soul
             The Verve                                                         Urban Hymns The Verve                                                   Urban Hymns
             The Verve                                                               Forth The Verve                                                         Forth
                   Ash                                                                   Trailer Ash                                                       Trailer
                   Ash

In [139]:
# Sanity-check: da nijedan naslov posle čišćenja i dalje ne završava izvođačevim imenom
still_dirty = check[check.apply(lambda r: r.album_title.endswith(' ' + r.artist_name), axis=1)]
print('Preostalo "prljavih" naslova:', len(still_dirty))
assert len(still_dirty) == 0

df = df.drop(columns=['album_title_original'])

Preostalo "prljavih" naslova: 0


## 2. Sečenje "đubreta" iz `allmusic_*_genre` / `allmusic_*_style` kolona

U koloni `allmusic_album_style` (i sličnim) posle stvarne, zarezom-razdvojene liste žanrova/stilova, sledi procureli tekst sa stranice (plejer kontrole, meniji, korisničke recenzije...), npr.:

```
"Alternative Pop/Rock, Alternative/Indie Rock, Dream Pop, Shoegaze, ... Listen on Amazon Listen on Spotify Set Your Streaming Service ..."
```

Zbog toga tražimo najraniju poziciju bilo kog od nekoliko karakterističnih "markera" koji uvek najavljuju početak procurelog teksta, i sve od te pozicije odsecamo.

In [140]:
JUNK_MARKERS = [
    'Listen on Amazon',
    'Set Your Streaming Service',
    'User Reviews',
    'Variations',
    'ADVERTISEMENT',
    'Also Performed By',
]

def strip_leaked_text(value):
    if pd.isna(value):
        return value
    cut = len(value)
    for marker in JUNK_MARKERS:
        idx = value.find(marker)
        if idx != -1:
            cut = min(cut, idx)
    return value[:cut].strip().rstrip(',').strip()

GENRE_STYLE_COLS = ['allmusic_album_genre', 'allmusic_album_style', 'allmusic_song_genre', 'allmusic_song_style']

before_max_len = {c: df[c].dropna().str.len().max() for c in GENRE_STYLE_COLS}

for c in GENRE_STYLE_COLS:
    df[c] = df[c].apply(strip_leaked_text)

after_max_len = {c: df[c].dropna().str.len().max() for c in GENRE_STYLE_COLS}

pd.DataFrame({'max_len_pre': before_max_len, 'max_len_posle': after_max_len})

,max_len_pre,max_len_posle
allmusic_album_genre,2533,27
allmusic_album_style,23034,137
allmusic_song_genre,34,34
allmusic_song_style,800,158


In [141]:
# Placeholder tekst "We currently don't have any Styles associated with this song." -> prazno (NaN)
placeholder_mask = df['allmusic_song_style'].str.contains('currently don', case=False, na=False)
print('Redova sa placeholder tekstom u allmusic_song_style:', placeholder_mask.sum())
df.loc[placeholder_mask, 'allmusic_song_style'] = np.nan

Redova sa placeholder tekstom u allmusic_song_style: 1240


## 3. `allmusic_album_theme` — leak iz gornjeg menija

Kod dela redova, `allmusic_album_theme` ne sadrži temu albuma, već je greškom scrape-ovan gornji navigacioni meni sajta (i praktično ceo ostatak stranice), koji uvek počinje istim fiksnim nizom kategorija:

```
"Blues Classical Country Electronic Folk International Pop/Rock Rap R&B Jazz Latin All Genres ..."
```

Detektujemo ove redove po tom fiksnom prefiksu i zamenjujemo vrednost sa `"Unknown"`.

In [142]:
MENU_LEAK_PREFIX = 'Blues Classical Country Electronic Folk International Pop/Rock'

menu_leak_mask = df['allmusic_album_theme'].str.startswith(MENU_LEAK_PREFIX, na=False)
print('Redova sa menu-leak vrednošću u allmusic_album_theme:', menu_leak_mask.sum())

df.loc[menu_leak_mask, 'allmusic_album_theme'] = 'Unknown'

Redova sa menu-leak vrednošću u allmusic_album_theme: 603


## 4. Rekonstrukcija zarezom-razdvojenih lista

Kolone `allmusic_composers`, `allmusic_song_style`, `allmusic_album_mood` i `allmusic_album_theme` povremeno sadrže više vrednosti razdvojenih **samo razmakom**, umesto zarezom, npr:

```
"Richard Ashcroft Simon Jones Nick McCabe Peter Salisbury"
```

umesto

```
"Richard Ashcroft, Simon Jones, Nick McCabe, Peter Salisbury"
```

Problem: same vrednosti (žanrovi, raspoloženja, teme, imena kompozitora) su same po sebi višerečne fraze ("Alternative/Indie Rock", "Nick McCabe"), pa prost split po razmaku ne radi.

Koristimo činjenicu da se u ovom istom datasetu iste vrednosti često javljaju i ispravno, zarezom-razdvojene (npr. `allmusic_album_style` je uvek zarezom razdvojen, `allmusic_song_mood`/`allmusic_song_theme` su skoro uvek zarezom razdvojeni, a i deo `allmusic_composers` redova je ispravan). Iz tih "čistih" vrednosti gradimo **rečnik poznatih pojmova** za svaku kategoriju (style, mood, theme, composer), i onda dati razmakom-razdvojeni tekst rekonstruišemo pomoću **greedy longest-match** algoritma: idemo kroz reči i na svakoj poziciji tražimo najduži mogući niz reči koji se poklapa sa nekim poznatim pojmom iz rečnika.

Dodatne heuristike:
- niz oblika `"X & Y"` (npr. `"Cool & Cocky"`) se uvek tretira kao jedna celina;
- token koji sadrži `"/"` (npr. `"Loss/Grief"`, `"Cynical/Sarcastic"`) je po konvenciji AllMusic-a uvek jedan pojam, pa se prihvata kao celina i kad nije nađen u rečniku;
- za `allmusic_composers` dodatno koristimo listu svih `artist_name` vrednosti iz dataseta (bend kao "kompozitor" je čest slučaj), a za preostale nerešene reči primenjujemo fallback heuristiku uparivanja reč-po-dve (ime + prezime), jer je to najčešći obrazac imena - ali tu vrednost posebno **označavamo** nižim nivoom pouzdanosti.

Kada rečnik i heuristike ne uspeju da u potpunosti rastave ceo tekst, **originalna vrednost se ne menja** (ne ubacujemo zareze nasumično) - takvi redovi se beleže za ručnu proveru.

In [143]:
def build_vocab(series):
    '''Skuplja pojedinačne pojmove iz VEĆ zarezom-razdvojenih vrednosti u seriji.'''
    vocab = set()
    for v in series.dropna():
        if ',' in v:
            for part in v.split(','):
                part = part.strip()
                if part:
                    vocab.add(part)
    return vocab


def merge_ampersand(tokens):
    '''Spaja obrazac 'X & Y' u jedan token.'''
    out = []
    i = 0
    while i < len(tokens):
        if i + 2 < len(tokens) and tokens[i + 1] == '&':
            out.append(tokens[i] + ' & ' + tokens[i + 2])
            i += 3
        else:
            out.append(tokens[i])
            i += 1
    return out


def reconstruct_list(value, vocab, max_ngram=6):
    '''Pokušava da razmakom-razdvojen tekst pretvori u zarezom-razdvojenu listu
    koristeći rečnik poznatih pojmova (greedy longest-match).
    Vraća (nova_vrednost, potpuno_resena: bool).'''
    if pd.isna(value) or value == 'Unknown':
        return value, True
    if ',' in value:
        return value, True  # već ispravno

    tokens = merge_ampersand(value.split())
    n = len(tokens)
    i = 0
    out = []
    fully_resolved = True

    while i < n:
        matched = False
        for L in range(min(max_ngram, n - i), 0, -1):
            candidate = ' '.join(tokens[i:i + L])
            if candidate in vocab:
                out.append(candidate)
                i += L
                matched = True
                break
        if matched:
            continue
        if '/' in tokens[i] and ' ' not in tokens[i]:
            # atomicna vrednost po AllMusic konvenciji (npr. "Loss/Grief")
            out.append(tokens[i])
            i += 1
        else:
            out.append(tokens[i])
            i += 1
            fully_resolved = False

    return ', '.join(out), fully_resolved

In [144]:
def apply_reconstruction(df, col, vocab, report_name):
    results = df[col].apply(lambda v: reconstruct_list(v, vocab))
    new_values = results.apply(lambda t: t[0])
    resolved = results.apply(lambda t: t[1])

    total_needed_fix = df[col].notna().sum()
    n_resolved = (resolved & df[col].notna()).sum()
    n_unresolved = (~resolved & df[col].notna()).sum()

    # Menjamo vrednost SAMO kad je rekonstrukcija potpuno uspela
    fix_mask = resolved & df[col].notna()
    df.loc[fix_mask, col] = new_values[fix_mask]

    print(f'[{report_name}] ukupno ne-null vrednosti: {total_needed_fix}')
    print(f'[{report_name}] potpuno rekonstruisano ili već ispravno: {n_resolved} ({n_resolved/total_needed_fix*100:.1f}%)')
    print(f'[{report_name}] ostalo nerešeno (vrednost NIJE menjana): {n_unresolved} ({n_unresolved/total_needed_fix*100:.1f}%)')
    print()

    unresolved_df = df.loc[~resolved & df[col].notna(), ['artist_name', 'album_title', 'song_title', col]].drop_duplicates(subset=[col])
    return unresolved_df

In [145]:
# Rečnici pojmova iz "čistih" kolona
style_vocab = build_vocab(df['allmusic_album_style']) | build_vocab(df['allmusic_song_style'])
mood_vocab = build_vocab(df['allmusic_song_mood']) | build_vocab(df['allmusic_album_mood'])
theme_vocab = build_vocab(df['allmusic_song_theme']) | build_vocab(df['allmusic_album_theme'])

print('Veličina rečnika: style =', len(style_vocab), '| mood =', len(mood_vocab), '| theme =', len(theme_vocab))

Veličina rečnika: style = 42 | mood = 172 | theme = 63


In [146]:
review_song_style = apply_reconstruction(df, 'allmusic_song_style', style_vocab, 'allmusic_song_style')
review_album_mood = apply_reconstruction(df, 'allmusic_album_mood', mood_vocab, 'allmusic_album_mood')
review_album_theme = apply_reconstruction(df, 'allmusic_album_theme', theme_vocab, 'allmusic_album_theme')

[allmusic_song_style] ukupno ne-null vrednosti: 1073
[allmusic_song_style] potpuno rekonstruisano ili već ispravno: 947 (88.3%)
[allmusic_song_style] ostalo nerešeno (vrednost NIJE menjana): 126 (11.7%)

[allmusic_album_mood] ukupno ne-null vrednosti: 1994
[allmusic_album_mood] potpuno rekonstruisano ili već ispravno: 1612 (80.8%)
[allmusic_album_mood] ostalo nerešeno (vrednost NIJE menjana): 382 (19.2%)

[allmusic_album_theme] ukupno ne-null vrednosti: 2313
[allmusic_album_theme] potpuno rekonstruisano ili već ispravno: 1531 (66.2%)
[allmusic_album_theme] ostalo nerešeno (vrednost NIJE menjana): 782 (33.8%)



### 4.1 Proširenje rečnika zvaničnim AllMusic tagovima

Rečnik iz prethodne ćelije je izgrađen isključivo iz ovog dataseta, pa ne pokriva celu AllMusic taksonomiju - deo vrednosti zato ostaje nerešen. Ispod dopunjujemo `style_vocab` i `theme_vocab` zvaničnim AllMusic style/theme tagovima koji se u ovom datasetu nikad nisu pojavili zarezom-razdvojeni (pa ih algoritam iz prethodne ćelije nije mogao "naučiti"), i ponovo pokrećemo rekonstrukciju samo za redove koji su ostali nerešeni.

Za `allmusic_album_mood` nije potreban rečnik: AllMusic Mood tagovi su uvek pojedinačne reči (ili slash-spojevi poput `"Amiable/Good-Natured"`), nikad višerečne fraze - pa je dovoljno svaki razmak zameniti zarezom.

In [147]:
EXTRA_STYLE_VOCAB = set('''British Psychedelia
Post-Grunge
Ambient Pop
Power Pop
Club/Dance
Punk
Punk/New Wave
Punk Revival
Heavy Metal
Dance-Pop
Instrumental Rock
Ska
Film Music
African Folk
Synth Pop
Jangle Pop
Hair Metal
Pop-Metal
Punk Metal
Soft Rock
Progressive Metal
Sludge Metal
Noise-Rock
Orchestral/Easy Listening
Garage Rock Revival
Blues-Rock
Singer/Songwriter
Trip-Hop
Arena Rock
Mod Revival
Blue-Eyed Soul
Adult Contemporary
New Wave
Neo-Soul
Roots Reggae
Folk-Rock
Sophisti-Pop
Chamber Pop
Glam Rock
College Rock
Folk Revival
Folk-Pop
Roots Rock
Pop'''.splitlines())

EXTRA_THEME_VOCAB = set('''Loneliness
Nighttime
At the Office
Open Road
Other Times & Places
Youth
Good Times
Vacation
Zeitgeist
YOLO
Word Play
Destiny
Drugs
Everyday Life
Lifecycle
Nostalgia
Wisdom
Country Life
Sweet Dreams
Starry Sky
Compassion
Daydreaming
Dreaming
Myths & Legends
Adventure
Passion
Nature
Visions
Sorrow
Longing
Friendship
Happiness
Relationships
Healing/Comfort
Background Music
Divorce
School
Heartbreak
Separation
Yearning
Sun
Dancing
Romantic Evening'''.splitlines())

style_vocab |= EXTRA_STYLE_VOCAB
theme_vocab |= EXTRA_THEME_VOCAB

# allmusic_album_mood: prost razmak -> zarez, bez potrebe za rečnikom
mood_still_broken = df['allmusic_album_mood'].notna() & ~df['allmusic_album_mood'].str.contains(',', na=False)
df.loc[mood_still_broken, 'allmusic_album_mood'] = df.loc[mood_still_broken, 'allmusic_album_mood'].apply(
    lambda v: ', '.join(v.split())
)

# Drugi prolaz rekonstrukcije sa proširenim rečnikom — menja samo redove koji do sada nisu bili rešeni
review_song_style = apply_reconstruction(df, 'allmusic_song_style', style_vocab, 'allmusic_song_style (2. prolaz)')
review_album_theme = apply_reconstruction(df, 'allmusic_album_theme', theme_vocab, 'allmusic_album_theme (2. prolaz)')

# Osvežavamo review_album_mood da odražava da je sada sve rešeno
review_album_mood = df.loc[
    df['allmusic_album_mood'].notna() & ~df['allmusic_album_mood'].str.contains(',', na=False),
    ['artist_name', 'album_title', 'song_title', 'allmusic_album_mood']
]

print('Preostalo nerešenih song_style:', len(review_song_style))
print('Preostalo nerešenih album_mood:', len(review_album_mood))
print('Preostalo nerešenih album_theme:', len(review_album_theme))

[allmusic_song_style (2. prolaz)] ukupno ne-null vrednosti: 1073
[allmusic_song_style (2. prolaz)] potpuno rekonstruisano ili već ispravno: 1073 (100.0%)
[allmusic_song_style (2. prolaz)] ostalo nerešeno (vrednost NIJE menjana): 0 (0.0%)

[allmusic_album_theme (2. prolaz)] ukupno ne-null vrednosti: 2313
[allmusic_album_theme (2. prolaz)] potpuno rekonstruisano ili već ispravno: 2313 (100.0%)
[allmusic_album_theme (2. prolaz)] ostalo nerešeno (vrednost NIJE menjana): 0 (0.0%)

Preostalo nerešenih song_style: 0
Preostalo nerešenih album_mood: 0
Preostalo nerešenih album_theme: 0


Za `allmusic_composers` koristimo isti princip, uz dva dodatka: rečnik proširujemo listom svih `artist_name` vrednosti (bend kao kompozitor), a za reči koje ni tada ne prepoznamo primenjujemo fallback uparivanje po dve reči (ime + prezime), uz posebnu kolonu koja beleži nivo pouzdanosti rekonstrukcije za svaki red.

In [148]:
composer_vocab = build_vocab(df['allmusic_composers']) | set(df['artist_name'].unique())

def reconstruct_composers(value, vocab, max_ngram=4):
    '''Vraća (nova_vrednost, nivo_pouzdanosti).
    Nivoi: 'original' (bilo je NaN ili već zarezom razdvojeno),
           'vocab_match' (u potpunosti rekonstruisano iz rečnika),
           'guessed_pairing' (deo rekonstruisan fallback uparivanjem reč+reč),
           'unresolved' (ostao bar jedan token koji nije mogao pouzdano da se spoji).'''
    if pd.isna(value):
        return value, 'original'
    if ',' in value:
        return value, 'original'

    tokens = merge_ampersand(value.split())
    n = len(tokens)
    i = 0
    out = []
    confidence = 'vocab_match'

    while i < n:
        matched = False
        for L in range(min(max_ngram, n - i), 0, -1):
            candidate = ' '.join(tokens[i:i + L])
            if candidate in vocab:
                out.append(candidate)
                i += L
                matched = True
                break
        if matched:
            continue
        if i + 1 < n:
            # fallback: pretpostavljamo "Ime Prezime"
            out.append(tokens[i] + ' ' + tokens[i + 1])
            i += 2
            if confidence != 'unresolved':
                confidence = 'guessed_pairing'
        else:
            out.append(tokens[i])
            i += 1
            confidence = 'unresolved'

    return ', '.join(out), confidence


comp_results = df['allmusic_composers'].apply(lambda v: reconstruct_composers(v, composer_vocab))
df['allmusic_composers'] = comp_results.apply(lambda t: t[0])
df['allmusic_composers_confidence'] = comp_results.apply(lambda t: t[1])

print(df['allmusic_composers_confidence'].value_counts())
print()
print('Redovi sa niskom pouzdanošću (za ručnu proveru):')
print(df.loc[df.allmusic_composers_confidence == 'unresolved', ['artist_name', 'song_title', 'allmusic_composers']].drop_duplicates())

allmusic_composers_confidence
original           1336
vocab_match         959
guessed_pairing      16
unresolved            2
Name: count, dtype: int64

Redovi sa niskom pouzdanošću (za ručnu proveru):
       artist_name                        song_title                                   allmusic_composers
513   Damon Albarn                         Mr. Tembo              Pentecostal City, Mission Church, Choir
1144   Paul Weller  Remember How We Started/Dominoes  Sigidi Abdullah, Harold Clayton, Mbaji Paul, Weller


In [149]:
# Objedinjena lista svih redova koji zahtevaju ručnu proveru (sekcija 4)
needs_review = pd.concat([
    review_song_style.assign(kolona='allmusic_song_style').rename(columns={'allmusic_song_style': 'vrednost'}),
    review_album_mood.assign(kolona='allmusic_album_mood').rename(columns={'allmusic_album_mood': 'vrednost'}),
    review_album_theme.assign(kolona='allmusic_album_theme').rename(columns={'allmusic_album_theme': 'vrednost'}),
], ignore_index=True)

print(f'Broj vrednosti koje su ostale nerešene nakon proširenja rečnika: {len(needs_review)}')

if len(needs_review) > 0:
    needs_review.to_csv('allmusic_needs_review.csv', index=False)
    print('Sačuvano u allmusic_needs_review.csv za ručnu proveru.')
needs_review.head(10)

Broj vrednosti koje su ostale nerešene nakon proširenja rečnika: 0


,artist_name,album_title,song_title,vrednost,kolona


## 5. Dodatne izmene

Tokom pregleda dataseta primećeno je još nekoliko manjih problema koji vredi rešiti pre spajanja sa ostalim podacima:

- kolone `spotify_id`, `allmusic_match_score`, `allmusic_match_type` su potpuno prazne, a `allmusic_match_status` je konstantna (`"pending"`) - trenutno ne nose nikakvu informaciju, pa ih uklanjamo;
- `allmusic_album_duration` meša format `MM:SS` i `HH:MM:SS` - dodajemo numeričku kolonu `allmusic_album_duration_sec` (trajanje u sekundama) radi lakšeg poređenja/sortiranja, uz zadržavanje originalnog teksta;
- generalni whitespace trim (npr. `allmusic_album_mood` je imao trailing razmake) na svim tekstualnim kolonama.

In [150]:
EMPTY_OR_CONSTANT_COLS = ['spotify_id', 'allmusic_match_score', 'allmusic_match_type', 'allmusic_match_status']
print('Uklanjam kolone:', EMPTY_OR_CONSTANT_COLS)
df = df.drop(columns=EMPTY_OR_CONSTANT_COLS)

Uklanjam kolone: ['spotify_id', 'allmusic_match_score', 'allmusic_match_type', 'allmusic_match_status']


In [151]:
def duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    parts = value.split(':')
    parts = [int(p) for p in parts]
    if len(parts) == 2:
        m, s = parts
        return m * 60 + s
    if len(parts) == 3:
        h, m, s = parts
        return h * 3600 + m * 60 + s
    return np.nan

df['allmusic_album_duration_sec'] = df['allmusic_album_duration'].apply(duration_to_seconds)
df[['allmusic_album_duration', 'allmusic_album_duration_sec']].drop_duplicates().head(10)

,allmusic_album_duration,allmusic_album_duration_sec
0,47:02,2822.0
10,01:03:59,3839.0
22,01:15:51,4551.0
35,01:04:18,3858.0
45,37:40,2260.0
56,01:02:23,3743.0
68,50:01,3001.0
80,47:29,2849.0
93,42:16,2536.0
104,51:27,3087.0


In [152]:
string_cols = df.select_dtypes(include=['object', 'string']).columns
for c in string_cols:
    df[c] = df[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

## 6. Uklanjanje URL kolona

Kolone `allmusic_artist_url`, `allmusic_song_url` i `allmusic_album_url` nisu potrebne za dalju analizu, pa ih uklanjamo.

In [153]:
URL_COLS = ['allmusic_artist_url', 'allmusic_song_url', 'allmusic_album_url']
print('Uklanjam kolone:', URL_COLS)
df = df.drop(columns=URL_COLS)

Uklanjam kolone: ['allmusic_artist_url', 'allmusic_song_url', 'allmusic_album_url']


## 7. `allmusic_track_pick` → boolean

Vrednosti `"yes"` / `"unknown"` pretvaramo u pravi `bool` (`True`/`False`) radi lakše analize (nema nedostajućih vrednosti u ovoj koloni).

In [154]:
df['allmusic_track_pick'] = df['allmusic_track_pick'].map({'yes': True, 'unknown': False})
assert df['allmusic_track_pick'].isnull().sum() == 0

print(df['allmusic_track_pick'].value_counts())
print('dtype:', df['allmusic_track_pick'].dtype)

allmusic_track_pick
False    1826
True      487
Name: count, dtype: int64
dtype: bool


## 8. `allmusic_release_date` -> format `YYYY-MM-DD`

Datumi dolaze u 3 različita formata: pun datum (`"July 3, 1995"`), samo mesec i godina (`"October, 1995"`) i samo godina (`"1996"`). Kada dan i/ili mesec nedostaju, ne izmišljamo ih - vrednost samo svodimo na onoliko preciznu koliko originalno jeste: `YYYY-MM-DD` za pun datum, `YYYY-MM` kad nedostaje dan, `YYYY` kad nedostaju i dan i mesec.

In [155]:
import datetime

def parse_release_date(value):
    if pd.isna(value):
        return np.nan
    value = value.strip()
    for fmt, out_fmt in (
        ('%B %d, %Y', '%Y-%m-%d'),  # pun datum, npr. "July 3, 1995"
        ('%B, %Y', '%Y-%m'),        # samo mesec i godina, npr. "October, 1995"
        ('%Y', '%Y'),               # samo godina, npr. "1996"
    ):
        try:
            dt = datetime.datetime.strptime(value, fmt)
            return dt.strftime(out_fmt)
        except ValueError:
            continue
    return np.nan  # nepoznat format - ne bi trebalo da se desi, proveravamo ispod

before = df['allmusic_release_date'].copy()
df['allmusic_release_date'] = df['allmusic_release_date'].apply(parse_release_date)

neparsirano = before.notna() & df['allmusic_release_date'].isna()
print('Redova koji nisu mogli da se parsiraju:', neparsirano.sum())
if neparsirano.sum() > 0:
    print(before[neparsirano].unique())

df[['allmusic_release_date']].drop_duplicates().head(10)

Redova koji nisu mogli da se parsiraju: 0


,allmusic_release_date
0,1993-06-21
10,1995-07-03
22,1997-09-30
35,2008-08-19
45,1995-10
56,1996-05-06
68,1998-10
80,2001-04-16
93,2004-06-29
104,2007-07-02


## 9. Ispravka `allmusic_composers` za konkretne pesme

Sledeće vrednosti su ručno provereno pogrešne (bile su označene kao `guessed_pairing` ili `unresolved` u koloni `allmusic_composers_confidence`) i zamenjujemo ih tačnim spiskom kompozitora.

Za sve ostale redove koji su bili `guessed_pairing`/`unresolved` je ručno potvrđeno da su tačni, pa ostaju nepromenjeni. Nakon ispravke, kolona `allmusic_composers_confidence` više nije potrebna i briše se.

In [156]:
COMPOSER_CORRECTIONS = {
    ('Shed Seven', 'Liquid Gold', 'Chasing Rainbows'): 'Paul Banks, Rick Witter',
    ('Paul Weller', 'Live Wood', 'Bull Rush/Magic Bus'): 'Pete Townshend, Paul Weller',
    ('Manic Street Preachers', 'Postcards from a Young Man', "Don't Be Evil"): 'Manic Street Preachers',
    ('Manic Street Preachers', 'Postcards from a Young Man', 'Some Kind of Nothingness'): 'Manic Street Preachers',
    ('Damon Albarn', 'Everyday Robots', 'Mr. Tembo'): 'Damon Albarn',
}

needs_check_mask = df['allmusic_composers_confidence'].isin(['guessed_pairing', 'unresolved'])

applied = 0
for (artist, album, song), correct_value in COMPOSER_CORRECTIONS.items():
    row_mask = (
        needs_check_mask
        & (df['artist_name'] == artist)
        & (df['album_title'] == album)
        & (df['song_title'] == song)
    )
    applied += row_mask.sum()
    df.loc[row_mask, 'allmusic_composers'] = correct_value

print('Broj ispravljenih redova:', applied)

df = df.drop(columns=['allmusic_composers_confidence'])

Broj ispravljenih redova: 6


## 10. Normalizacija `allmusic_recording_location`

Kolona ima dva odvojena problema:

1. Oko 11 vrednosti uopšte nisu lokacije, već **datumi/periodi snimanja** koji su greškom završili u ovoj koloni (npr. `"1994"`, `"2001 - 2002"`, `"July 19, 2007"`) - tretiramo ih kao nedostajući podatak.
2. Veliki broj preostalih vrednosti su **varijante istog mesta** - različit nivo detalja (npr. `"Rockfield"` / `"Rockfield, Wales"`, `"Atomic Heart"` / `"Atomic Heart Studios, NYC"` / `"Atomic Heart Studios, New York City"`) - spajamo ih u jedan kanonski zapis po grupi.

In [157]:
GARBAGE_LOCATION_VALUES = {
    '13', '1994', '1996', '2001 - 2002',
    'April, 1999 - August, 1999', 'August, 1998 - February, 1999',
    'December, 1993 - April, 1994', 'July 19, 2007',
    'June, 1998 - October, 1998', 'May 9, 2007', 'October 20, 2007',
}

garbage_mask = df['allmusic_recording_location'].isin(GARBAGE_LOCATION_VALUES)
print('Redova sa datumom umesto lokacije:', garbage_mask.sum())
df.loc[garbage_mask, 'allmusic_recording_location'] = np.nan

LOCATION_NORMALIZATION = {
    'Rockfield': 'Rockfield, Wales',
    'Atomic Heart': 'Atomic Heart Studios, New York City',
    'Atomic Heart Studios, NYC': 'Atomic Heart Studios, New York City',
    'Big Mushroom': 'Big Mushroom Studios, Cheshire, England',
    'Big Mushroom Studios, Cheshire': 'Big Mushroom Studios, Cheshire, England',
    'Big Mushroom, Cheshire': 'Big Mushroom Studios, Cheshire, England',
    'Black Barn': 'Black Barn Studios, England',
    'Black Barn Studios': 'Black Barn Studios, England',
    'Black Barn, England': 'Black Barn Studios, England',
    'Door To The River Studios, Wales': 'Door to the River Studios, Wales',
    'Door to River': 'Door to the River Studios, Wales',
    'Faster Recording Studio, Cardiff': 'Faster Studios, Cardiff, Wales',
    'Faster Studios, Wales': 'Faster Studios, Cardiff, Wales',
    'Grouse Lodge Studios Ireland': 'Grouse Lodge Studios, Ireland',
    'Master Rock': 'Master Rock Studios',
    'Rak Studios': 'RAK Studios, London, England',
    'Rak Studios, London, England': 'RAK Studios, London, England',
    'RAK Recording Studios': 'RAK Studios, London, England',
    'Monnow Valley Studios, Monmouth, Gwent': 'Monnow Valley Studios, Monmouth, Wales',
    'Monnow Valley Studios, Monmouth, South Wales': 'Monnow Valley Studios, Monmouth, Wales',
    'Big Noise Recorders, Wales': 'Big Noise Recorders, Cardiff, Wales',
    'Sawmills Studio': 'Sawmills Studio, Golant, Cornwall, England',
    'Sawmills Studio, Golant, Forey, Cornwall, England': 'Sawmills Studio, Golant, Cornwall, England',
    'Cts Studios': 'CTS Studios, Whitfield St., London, England',
    'Abbet Road Studios, London, England': 'Abbey Road Studios, London, England',
    'AIR Studios': 'Air Studios, London, England',
}

n_changed = df['allmusic_recording_location'].isin(LOCATION_NORMALIZATION.keys()).sum()
df['allmusic_recording_location'] = df['allmusic_recording_location'].replace(LOCATION_NORMALIZATION)

print('Redova normalizovano na kanonski naziv:', n_changed)
print('Broj jedinstvenih lokacija pre/posle: 109 ->', df['allmusic_recording_location'].dropna().nunique())

Redova sa datumom umesto lokacije: 156
Redova normalizovano na kanonski naziv: 362
Broj jedinstvenih lokacija pre/posle: 109 -> 78


## 11. Popunjavanje nedostajućih vrednosti sa `"Unknown"`

Sve preostale null vrednosti u **tekstualnim** kolonama popunjavamo sa `"Unknown"` radi lakše dalje analize (grupisanje, filtriranje i sl. bez posebnog tretmana NaN-a).

Numerička kolona `allmusic_album_duration_sec` namerno ostaje van ovog koraka - puniti je stringom `"Unknown"` bi joj promenilo tip iz broja u tekst i onemogućilo dalju numeričku analizu (npr. prosečno trajanje pesme), pa tu nedostajuće vrednosti ostaju kao `NaN`.

In [158]:
object_cols = df.select_dtypes(include=['object', 'string']).columns
before_nulls = df[object_cols].isnull().sum()
print('Null vrednosti po tekstualnoj koloni pre popunjavanja:')
print(before_nulls[before_nulls > 0])

df[object_cols] = df[object_cols].fillna('Unknown')

print()
print('Ukupno popunjeno:', before_nulls.sum())
print('Preostalo null vrednosti u celom datasetu (očekivano: samo allmusic_album_duration_sec):')
print(df.isnull().sum()[df.isnull().sum() > 0])

Null vrednosti po tekstualnoj koloni pre popunjavanja:
allmusic_song_genre            1241
allmusic_song_style            1240
allmusic_song_mood             1230
allmusic_song_theme            1477
allmusic_composers              440
allmusic_album_style            109
allmusic_album_mood             319
allmusic_album_duration         189
allmusic_recording_location     924
dtype: int64

Ukupno popunjeno: 7169
Preostalo null vrednosti u celom datasetu (očekivano: samo allmusic_album_duration_sec):
allmusic_album_duration_sec    189
dtype: int64


## 12. Finalna provera i izvoz

In [159]:
print('Dimenzije nakon čišćenja:', df.shape)
print()
print('Duplikati (ceo red):', df.duplicated().sum())
print()
print('Primer očišćenih naslova albuma:')
print(df[['artist_name', 'album_title']].drop_duplicates().head(10).to_string(index=False))

Dimenzije nakon čišćenja: (2313, 18)

Duplikati (ceo red): 0

Primer očišćenih naslova albuma:
artist_name               album_title
  The Verve         A Storm in Heaven
  The Verve           A Northern Soul
  The Verve               Urban Hymns
  The Verve                     Forth
        Ash                   Trailer
        Ash                      1977
        Ash           Nu-Clear Sounds
        Ash           Free All Angels
        Ash                  Meltdown
        Ash Twilight of the Innocents


In [160]:
print('Primer očišćenih vrednosti:')
sample_cols = ['artist_name', 'album_title', 'allmusic_album_style', 'allmusic_album_mood',
               'allmusic_album_theme', 'allmusic_composers', 'allmusic_track_pick',
               'allmusic_release_date', 'allmusic_recording_location']
df[sample_cols].sample(5, random_state=42)

Primer očišćenih vrednosti:


,artist_name,album_title,allmusic_album_style,allmusic_album_mood,allmusic_album_theme,allmusic_composers,allmusic_track_pick,allmusic_release_date,allmusic_recording_location
1646,Shed Seven,Where Have You Been Tonight? Live,"Alternative Pop/Rock, Alternative/Indie Rock, British Trad Rock, Britpop","Rousing, Stylish",Unknown,"Paul Banks, Tom Gladwin, Alan Leach, Rick Witter",False,2003-06-24,Unknown
509,Damon Albarn,Dr. Dee,"Alternative Singer/Songwriter, Alternative/Indie Rock, Opera","Austere, Dark, Literate, Plaintive, Reflective, Wintry, Wistful, Dramatic, Mysterious, Regretful, Somber, Sophisticated, Theatrical, Witty, Wry, Mystical","Visions, Winter","Alastair Merry, Alexander Overington, Amy Payne, Andrew Walters, André de Ridder, Anna Dennis, Anne Allen, Ashley Catling, BBC Philharmonic Orchestra, Belin...",False,2012-05-07,MediaCityUK
1882,Supergrass,I Should Coco,"Alternative Pop/Rock, Alternative/Indie Rock, Britpop, Pop Punk","Boisterous, Effervescent, Giddy, Gleeful, Irreverent, Rollicking, Energetic, Fun, Playful, Quirky",Unknown,Supergrass,True,1995-07-18,"Sawmills Studio, Golant, Cornwall, England"
44,The Verve,Forth,"Alternative Pop/Rock, Alternative/Indie Rock, Adult Alternative Pop/Rock","Autumnal, Hypnotic, Laid-Back/Mellow, Plaintive, Poignant, Reflective, Bittersweet, Druggy, Earnest, Ethereal, Intimate, Melancholy, Provocative, Rousing, S...","Autumn, Introspection, Rainy Day, Reflection, Relaxation","Richard Ashcroft, Cameron Jenkins, Chris Potter, Davide Rossi, Dean Chalkley, Jazz Summers, Nick McCabe, Peter Salisbury, Simon Jones, Tim Bran, Tim Parry",False,2008-08-19,"State Of The Ark Studios, London, England"
1586,Pulp,Live!,Unknown,Unknown,Unknown,Unknown,False,2026-08-28,Unknown


In [161]:
df.to_csv('allmusic_clean.csv', index=False)
print('Sačuvano: allmusic_clean.csv', df.shape)

Sačuvano: allmusic_clean.csv (2313, 18)
